# 03 · Turn an estimate into a bounded decision

Reproduce the saved demonstration with the shared decision engine. PostgreSQL must be running. All revenue changes below are simulated, not realized business improvements.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
from retail_pricing.config import DATA, ARTIFACTS, REPORTS
observations = pd.read_parquet(DATA / "processed/observations.parquet")
products = pd.read_parquet(DATA / "processed/products.parquet")
quality = json.loads((REPORTS / "data_quality.json").read_text())

from retail_pricing import db
from retail_pricing.service import PricingService
from retail_pricing.schemas import ScenarioRequest, RecommendationRequest
service = PricingService(db.engine())
demo = json.loads((REPORTS / "demo_result.json").read_text())
product_id = demo["scenario"]["product_id"]
store_id = demo["scenario"]["store_id"]
context = service.analysis(product_id, store_id)["context"]
display(pd.Series(context).to_frame("value"))

,value
baseline_price,2.95
reference_price,2.95
baseline_support,"{'feature': 0, 'display': 0, 'tpr_only': 0}"
last_observed_week,2012-01-04
price_support,"[2.93, 4.49]"
support_combinations,"[{'feature': 0, 'display': 0, 'tpr_only': 0}, ..."
next_week,2012-01-11
warnings,[]


## Reproduce the baseline and proposal
The week, store and seasonal conditions are held fixed. Costs are absent unless explicitly supplied. The model version identifies the fit used for the saved result.

In [2]:
assumptions = demo["scenario"]["assumptions"]
fields = ScenarioRequest.model_fields
request = ScenarioRequest(**{key: value for key, value in assumptions.items() if key in fields})
comparison = service.simulate(request, persist=False)
assert comparison["model_version"] == demo["scenario"]["model_version"]
assert np.isclose(comparison["proposed"]["revenue"], demo["scenario"]["proposed"]["revenue"])
display(pd.DataFrame([comparison["baseline"], comparison["proposed"]], index=["baseline", "proposed"]))
display(pd.Series(comparison["changes"]).to_frame("simulated change"))

,price,units,revenue,units_interval,revenue_interval,contribution_profit,margin_pct
baseline,2.95,9.779795,28.850397,"[8.978467844574197, 10.65264143459446]","[26.486480141493885, 31.425292232053657]",None,None
proposed,3.25,11.051047,35.915902,"[9.102171640817883, 13.417197812856738]","[29.58205783265812, 43.6058928917844]",None,None


,simulated change
units_pct,12.998753
revenue_pct,24.490152
revenue,7.065506


In [3]:
px.line(pd.DataFrame(comparison["curve"]), x="price", y="revenue", markers=True, title="Revenue curve at proposed support settings").show(renderer="plotly_mimetype")
print(comparison["disclaimer"])

Observational model estimates, not causal effects or realized gains. Intervals describe conditional mean uncertainty, not a range of future sales.


## Search the feasible grid
The grid is small enough to enumerate. A candidate may improve modeled revenue but fail a margin constraint. A boundary result is expected for many constant-elasticity models.

In [4]:
recommendation = service.recommend(RecommendationRequest(product_id=product_id, store_id=store_id))
print("Status:", recommendation["status"])
display(pd.DataFrame(recommendation["candidates"]).head(10))
print(recommendation["note"])

Status: recommended


,price,units,revenue,units_interval,revenue_interval,contribution_profit,margin_pct,support,unchanged_baseline,promotion_cost
0,3.25,11.051047,35.915902,"[9.102171640817877, 13.417197812856733]","[29.5820578326581, 43.60589289178438]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
1,3.22,11.112599,35.782567,"[9.159057910292642, 13.482810927721474]","[29.49216647114231, 43.414651187263146]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
2,3.19,11.175074,35.648486,"[9.21673788249008, 13.54950961198146]","[29.401393845143353, 43.222935662220856]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
3,3.16,11.238496,35.513647,"[9.275228373663293, 13.617324269825946]","[29.309721660776006, 43.03074469264999]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
4,3.13,11.302888,35.378040,"[9.334546678568568, 13.686286481228585]","[29.217131103919616, 42.83807668624547]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
5,3.10,11.368275,35.241652,"[9.394710588219738, 13.756429058804958]","[29.12360282348119, 42.644930082295375]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
6,3.07,11.434681,35.104470,"[9.455738408519348, 13.82778610792306]","[29.029116914154397, 42.451303351323794]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
7,3.04,11.502133,34.966484,"[9.517648979824862, 13.900393090280415]","[28.93365289866758, 42.25719499445246]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
8,3.01,11.570658,34.827680,"[9.580461697512833, 13.974286891177293]","[28.837189709513623, 42.06260354244365]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0
9,2.98,11.640283,34.688044,"[9.644196533609438, 14.049505890732622]","[28.739705670156127, 41.867527554383216]",None,None,"{'feature': 0, 'display': 1, 'tpr_only': 0}",False,0


Best within the tested price grid. Constant elasticity often selects a boundary; this is not a universal optimum.


## A deliberate infeasible case
The following is a clearly labeled hypothetical high cost, not a value from the dataset. The tool should report no feasible recommendation rather than relaxing the constraint.

In [5]:
high_cost = context["price_support"][1] * 2
infeasible = service.recommend(RecommendationRequest(product_id=product_id, store_id=store_id, unit_cost=high_cost, min_margin_pct=20))
assert infeasible["status"] == "infeasible"
print(infeasible["status"], "with assumed unit cost", high_cost)

infeasible with assumed unit cost 8.98


## Decision memo
Explain the proposed price/support, the allowed price range, simulated revenue change, uncertainty and assumptions. Do not equate support associations with experimental lift. A controlled trial and inventory checks would be needed before operational use.